<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

In [ ]:
!pip install gradio matplotlib -q

In [ ]:
import gradio as gr
import pandas as pd
import gspread
import datetime
import matplotlib.pyplot as plt
from google.colab import auth
from google.auth import default

In [ ]:
# 1. 安裝中文字體
!sudo apt-get install -y fonts-noto-cjk

# 2. 清除 matplotlib 字體快取
!rm -rf ~/.cache/matplotlib

# 3. 強制讓 matplotlib 讀取新安裝的字體檔案
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 搜尋系統中的 Noto Sans CJK 字體檔案並加入 FontManager
font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/opentype/noto'])
for font_file in font_files:
    fm.fontManager.addfont(font_file)

# 4. 配置使用中文字體
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP']
plt.rcParams['axes.unicode_minus'] = False

print("字體重新載入完成，請執行下方的測試區塊。")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  fonts-noto-cjk-extra
The following NEW packages will be installed:
  fonts-noto-cjk
0 upgraded, 1 newly installed, 0 to remove and 5 not upgraded.
Need to get 61.2 MB of archives.
After this operation, 93.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-cjk all 1:20220127+repack1-1 [61.2 MB]
Fetched 61.2 MB in 5s (12.1 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fon

In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SHEET_URL = 'https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing'
gsheets = gc.open_by_url(SHEET_URL)
worksheet = gsheets.worksheet('工作表1')

In [ ]:
# 3. 功能函式定義

def get_raw_data():
    """讀取試算表原始資料並回傳 DataFrame"""
    data = worksheet.get_all_values()
    # 將最後一欄替換為「實付金額」以便後續處理
    col_names = ["日期", "時間", "分類", "品項", "金額", "地點", "支付方式", "付款人/分攤成員", "實付金額"]

    if len(data) <= 1:
        return pd.DataFrame(columns=col_names)

    df = pd.DataFrame(data[1:], columns=data[0])

    # 強制更新 DataFrame 的欄位名稱
    if len(df.columns) == 9:
        df.columns = col_names

    return df

def get_pie_chart(start_date=None, end_date=None):
    """讀取試算表資料，根據日期區間過濾，並生成圓餅圖 (依實付金額)"""
    df = get_raw_data()
    fig, ax = plt.subplots(figsize=(6, 6))

    if df.empty:
        ax.text(0.5, 0.5, "尚無資料", ha='center', va='center', fontsize=12)
        ax.axis('off')
        return fig

    # 1. 處理日期轉換與過濾
    df['日期格式'] = pd.to_datetime(df['日期'], errors='coerce')

    if start_date:
        try:
            sd = pd.to_datetime(start_date)
            df = df[df['日期格式'] >= sd]
        except Exception:
            pass # 若日期格式錯誤則略過該條件

    if end_date:
        try:
            ed = pd.to_datetime(end_date)
            df = df[df['日期格式'] <= ed]
        except Exception:
            pass

    # 2. 處理金額並繪製圖表
    df['實付金額'] = pd.to_numeric(df['實付金額'], errors='coerce').fillna(0)
    category_sums = df.groupby('分類')['實付金額'].sum()

    # 如果該區間沒有任何支出金額
    if category_sums.sum() == 0:
        ax.text(0.5, 0.5, "此日期區間無支出紀錄", ha='center', va='center', fontsize=12)
        ax.axis('off')
        ax.set_title('支出分類佔比 (依實付金額計算)')
        return fig

    category_sums.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=140)
    ax.set_ylabel('')

    # 動態設定標題顯示日期區間
    title = '支出分類佔比 (依實付金額計算)'
    if start_date or end_date:
        title += f'\n({start_date or "最早"} ~ {end_date or "最新"})'
    ax.set_title(title)

    return fig

def submit_to_sheet(date_str, time_str, category, item, amount, location, payment_method, split_count, payer_info, filter_start, filter_end):
    """寫入資料並返回狀態與更新後的圖表及表格"""
    try:
        datetime.datetime.strptime(date_str, '%Y-%m-%d')

        # AA 拆帳計算
        split_count = int(split_count) if split_count and split_count > 0 else 1
        amount = float(amount) if amount else 0.0
        final_amount = round(amount / split_count) # 實付金額

        # 整理備註資訊
        if split_count > 1:
            member_note = f"{payer_info} ({split_count}人AA)" if payer_info else f"{split_count}人AA"
        else:
            member_note = payer_info

        # 第 5 欄填入「總金額」，第 9 欄填入「實付金額」
        new_row = [[date_str, time_str, category, item, str(amount), location, payment_method, member_note, str(final_amount)]]
        worksheet.append_rows(values=new_row, value_input_option='USER_ENTERED')

        # 這裡一併帶入目前的篩選日期，讓寫入後圖表不會跑掉
        new_plot = get_pie_chart(filter_start, filter_end)
        new_df = get_raw_data()

        # 狀態訊息提示
        if split_count > 1:
            status_msg = f"成功寫入！{item} (總價 ${amount}，實付 ${final_amount})"
        else:
            status_msg = f"成功寫入！{item} (實付 ${final_amount})"

        return status_msg, new_plot, new_df
    except Exception as e:
        return f"發生錯誤：{str(e)}", None, None

# 4. Gradio 介面配置
with gr.Blocks() as demo:
    gr.Markdown("# 日常支出速算與視覺化")

    with gr.Tab("支出紀錄"):
        date_in = gr.Textbox(label="日期 (YYYY-MM-DD)", value=datetime.date.today().isoformat())
        time_in = gr.Textbox(label="時間", value=datetime.datetime.now().strftime("%H:%M"))
        cat_in = gr.Dropdown(choices=["食物", "飲料", "交通", "書", "其他"], label="支出類別", value="食物")
        item_in = gr.Textbox(label="品項")

        with gr.Row():
            amt_in = gr.Number(label="總金額")
            split_count_in = gr.Number(label="拆帳人數", value=1, minimum=1, precision=0)

        location_in = gr.Textbox(label="地點")
        payment_in = gr.Dropdown(choices=["現金", "信用卡", "電子支付", "轉帳"], label="支付方式", value="現金")
        payer_in = gr.Textbox(label="付款人/分攤成員", placeholder="例如：先代墊的小明，或是與誰平分")

        btn = gr.Button("記錄支出", variant="primary")
        status = gr.Textbox(label="狀態")

    with gr.Tab("支出分析圖"):
        with gr.Row():
            # 取得今天日期與當月1號
            today = datetime.date.today()
            first_day = today.replace(day=1)
            filter_start = gr.Textbox(label="開始日期 (YYYY-MM-DD)", value=first_day.isoformat())
            filter_end = gr.Textbox(label="結束日期 (YYYY-MM-DD)", value=today.isoformat())

        refresh_chart_btn = gr.Button("依日期區間重新繪製圖表", variant="secondary")
        plot_output = gr.Plot(label="支出分類佔比")

    with gr.Tab("檢視原始資料"):
        data_table = gr.DataFrame(label="Google Sheet 原始資料")
        refresh_table_btn = gr.Button("重新整理資料表")

    # 事件綁定 (提交表單時也要參考目前的日期過濾區間)
    btn.click(
        fn=submit_to_sheet,
        inputs=[date_in, time_in, cat_in, item_in, amt_in, location_in, payment_in, split_count_in, payer_in, filter_start, filter_end],
        outputs=[status, plot_output, data_table]
    )
    refresh_chart_btn.click(
        fn=get_pie_chart,
        inputs=[filter_start, filter_end],
        outputs=plot_output
    )
    refresh_table_btn.click(fn=get_raw_data, outputs=data_table)

    # 啟動時預載 (帶入預設篩選時間)
    demo.load(fn=get_pie_chart, inputs=[filter_start, filter_end], outputs=plot_output)
    demo.load(fn=get_raw_data, outputs=data_table)

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc9cdbe63cfbd00abf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fc9cdbe63cfbd00abf.gradio.live
